<a href="https://colab.research.google.com/github/Hwk040319/MJY-ML-admin/blob/main/test_private.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 배터리 열폭주 이미지 분류 · 비공개 Test 채점 (운영진 전용)

**이 노트북은 운영진 전용입니다.** 참가자에게 공유하지 마세요.

**사용법 요약**: 내 드라이브에 `MJY` 폴더를 만들고 그 안에 팀별 `best_model.pt`를
전부 모아둔 다음, 이 노트북에서 **런타임 → 모두 실행**을 누르세요. 드라이브 연결/정답 파일
업로드처럼 한 번만 확인하면 되는 부분만 화면에 뜨는 대로 처리해주면, 나머지는 자동으로 끝까지
실행되면서 `MJY` 폴더 안 모든 팀의 Macro F1이 순서대로 출력됩니다.

## 0. GPU 확인

In [ ]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
# False 면 런타임 -> 런타임 유형 변경 -> T4 GPU 선택 후 이 셀 다시 실행

## 1. 채점 코드 받기

In [ ]:
!git clone https://github.com/Hwk040319/MJY-ML-admin.git
%cd MJY-ML-admin
!pip install -q -r requirements.txt
print('채점 코드 준비 완료')

## 2. 원본 데이터(비공개 Test 포함) 압축 풀기

실행 전에 구글 드라이브의 `battery-colab-grouped.tar`를 본인 드라이브에 **사본으로 복사**해두고,
파일 이름에서 "의 사본"을 지워 원래 이름과 똑같이 맞춰두세요 (내 드라이브 맨 위/루트에 있어야 합니다).
이 셀 실행 중 드라이브 접근 허용 창이 뜨면 허용해주세요 (세션당 한 번만 뜹니다).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/battery-project-grouped
!tar -xf "/content/drive/MyDrive/battery-colab-grouped.tar" -C /content/battery-project-grouped
!ls /content/battery-project-grouped/participant/data_grouped
# private_test, public_val, train 세 폴더가 보이면 정상입니다.

## 3. 비공개 정답 파일(`private_labels.csv`) 준비

이 파일은 이 저장소에도, 위 tar 안에도 들어있지 않습니다 — 정답 유출 방지를 위해 운영진(hwk040319)이
카톡/이메일 등 별도 경로로 직접 전달하는 파일입니다. 전달받은 `private_labels.csv`를 아래 셀로
업로드하세요. (세션당 한 번만 하면 됩니다. "런타임 → 모두 실행"으로 돌리는 중이라면 이 셀에서
잠깐 멈추고 파일 선택 창이 뜹니다 — 파일을 고르면 이어서 자동으로 진행됩니다.)

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()   # 운영진에게 받은 private_labels.csv 선택
shutil.move(list(uploaded.keys())[0],
            "/content/battery-project-grouped/participant/data_grouped/private_test/private_labels.csv")
print('정답 파일 배치 완료')

## 4. 팀별 Macro F1 한 번에 채점

**미리 준비**: 내 드라이브의 `MJY` 폴더(`/content/drive/MyDrive/MJY`) 안에 팀별 체크포인트를
전부 넣어두세요. 파일 이름은 팀을 구분할 수 있게 자유롭게 지으면 됩니다 (예: `1조.pt`,
`2조_best_model.pt`, 또는 팀별 하위 폴더 `1조/best_model.pt`처럼 넣어도 됩니다 — 하위 폴더까지
전부 찾아서 채점합니다).

이 셀은 그 폴더 안의 `.pt` 파일을 전부 찾아 하나씩 채점합니다. 새 팀 파일을 추가했으면 이 셀만
다시 실행하면 됩니다.

In [ ]:
from pathlib import Path

ckpt_dir = Path("/content/drive/MyDrive/MJY")
ckpt_files = sorted(p for p in ckpt_dir.rglob("*.pt") if p.is_file())

if not ckpt_files:
    print(f"체크포인트(.pt)를 찾을 수 없습니다: {ckpt_dir}")
    print("이 폴더 안에 팀별 best_model.pt 파일을 넣은 뒤 이 셀을 다시 실행하세요.")
else:
    print(f"{len(ckpt_files)}개 체크포인트 발견, 순서대로 채점합니다.\n")
    for ckpt in ckpt_files:
        print(f"\n{'='*20} {ckpt.relative_to(ckpt_dir)} {'='*20}")
        !python score_checkpoint.py \
            --checkpoint "{ckpt}" \
            --data-root /content/battery-project-grouped/participant/data_grouped

## 주의사항

- 최종 순위·등수 계산 방식은 아직 확정 전입니다. 지금은 팀별 `Macro F1` 값만 기록해두세요.
- 채점에 쓰는 데이터(특히 `private_test`, `private_labels.csv`)와 이 노트북·저장소는 참가자에게 절대 공유·유출하면 안 됩니다.
- Colab 세션이 끊기면 1~3번부터 다시 실행하면 됩니다 (드라이브 `MJY` 폴더 내용은 그대로 남아있으니 4번은 바로 다시 돌아갑니다).
- 팀을 나중에 더 추가하고 싶으면, 드라이브 `MJY` 폴더에 파일만 추가하고 4번 셀을 다시 실행하면 됩니다.